In [ ]:
# Cell 1 — Setup + data load: BTCUSDT 30m panel (full history, tz-naive UTC)
import pandas as pd
from backtest.data import DataPanel

RAW = pd.read_parquet(r"C:\Users\User\backtest_engine\backtest_engine2\data\btc_30m.parquet")

prices = RAW.pivot(index="open_time", columns="symbol", values="close")
volume = RAW.pivot(index="open_time", columns="symbol", values="quote_volume")
prices.index = prices.index.tz_localize(None)   # tz-naive UTC (values unchanged)
volume.index = volume.index.tz_localize(None)

panel = DataPanel(prices, volume=volume, check_outliers=True)

n_days = len(pd.unique(prices.index.date))
expected = (prices.index[-1] - prices.index[0]) / pd.Timedelta("30min") + 1
print(f"Bars: {len(panel.dates)}  (expected on full 30m grid: {int(expected)}, gap {1 - len(panel.dates)/int(expected):.3%})")
print(f"Assets: {len(panel.assets_all)}  -> {list(panel.assets_all)}")
print(f"Range: {panel.dates[0]} -> {panel.dates[-1]}  ({n_days} UTC days)")
print(f"NaN fraction in prices: {prices.isna().mean().mean():.3%}")
prices.tail(3)


In [ ]:
# Cell 2 — Stage 1 amendment: re-stamp bars to close-time (open_time + 30min).
# Bar labeled t now means "all information in this bar existed by t" — removes
# the 30-minute lookahead DataView would otherwise grant on every bar.
# Outliers were reviewed and accepted in Cell 1 (real moves), so check_outliers=False here.
prices = prices.copy()
volume = volume.copy()
prices.index = prices.index + pd.Timedelta("30min")
volume.index = volume.index + pd.Timedelta("30min")

panel = DataPanel(prices, volume=volume, check_outliers=False)

print(f"Bars: {len(panel.dates)} (unchanged from Cell 1: 154273)")
print(f"Range: {panel.dates[0]} -> {panel.dates[-1]}")
print("Midnight-stamped bars (= 23:30->24:00 bars, the last-half-hour P&L bar):",
      int(((panel.dates.hour == 0) & (panel.dates.minute == 0)).sum()))
print("00:30-stamped bars (= first-half-hour signal bars):",
      int(((panel.dates.hour == 0) & (panel.dates.minute == 30)).sum()))
prices.tail(2)


In [ ]:
# Cell 3 — Stage 2: paper-exact intraday TSM strategy.
# Signal: sign of first-half-hour return (00:00->00:30 UTC, close-time stamps).
# Trade: hold sign*gross over the last half-hour only (enter 23:30 bar, flat at 00:00 bar).
# 00:30 is a safety-exit bar: normally already flat -> no trade, no cost; it only
# matters on days where the midnight bar is missing from the grid.
import numpy as np
from backtest.strategy import Strategy
from backtest.risk import RiskConfig


def _itsm_rebalance_bars(dates):
    hm = dates.hour * 60 + dates.minute
    return dates[(hm == 23 * 60 + 30) | (hm == 0) | (hm == 30)]


class IntradayTSM(Strategy):
    """Gao/Han/Li/Zhou (JFE 2018) construction on BTC: first half-hour predicts last half-hour."""
    rebalance_frequency = staticmethod(_itsm_rebalance_bars)

    def __init__(self, gross: float = 0.5):
        self.gross = gross
        self.risk = RiskConfig(max_position=gross, max_gross=1.0)  # fields: 03-risk.md §2

    def __repr__(self):
        return f"IntradayTSM(signal=00:00-00:30, trade=23:30-24:00, gross={self.gross})"

    def generate_weights(self, data, t):
        zero = pd.Series(0.0, index=data.assets)
        if not (t.hour == 23 and t.minute == 30):
            return zero                                      # exit / safety bars: target flat
        day = t.normalize()
        px = data.prices["BTCUSDT"]
        try:
            p_mid = px.at[day]                               # price at 00:00 (close-time stamp)
            p_0030 = px.at[day + pd.Timedelta("30min")]      # price at 00:30
        except KeyError:
            return zero                                      # signal bars missing -> no trade
        if pd.isna(p_mid) or pd.isna(p_0030) or p_0030 == p_mid:
            return zero
        return pd.Series({"BTCUSDT": self.gross * np.sign(p_0030 / p_mid - 1.0)})


strat = IntradayTSM(gross=0.5)
print(strat)
rb = strat.rebalance_dates(panel.dates)
years = (panel.dates[-1] - panel.dates[0]).days / 365.25
print(f"Rebalance bars: {len(rb)} total, {len(rb) / years:.0f}/yr (expect ~3/day ~= 1096/yr)")
print(f"  entry bars (23:30): {int(((rb.hour == 23) & (rb.minute == 30)).sum())}")
print(f"  exit bars  (00:00): {int(((rb.hour == 0) & (rb.minute == 0)).sum())}")
print(f"  safety     (00:30): {int(((rb.hour == 0) & (rb.minute == 30)).sum())}")


In [ ]:
# Cell 4 — Stage 1 amendment: volume from quote (USD) to base units (BTC).
# MarketImpact/LiquidityCap compute dollar-ADV as volume * price (components.py:65-76,
# liquidity.py:22-39), so the panel's volume must be in base units, not dollars.
volume = volume / prices  # BTC per 30m bar; indexes/columns identical by construction

panel = DataPanel(prices, volume=volume, check_outliers=False)

implied_dollar_adv = (volume["BTCUSDT"].tail(20).median() * prices["BTCUSDT"].iloc[-1])
raw_quote_median = RAW["quote_volume"].tail(20).median()
print(f"Implied $ per 30m bar (volume*price, last 20 bars): {implied_dollar_adv:,.0f}")
print(f"Raw quote_volume median        (last 20 bars): {raw_quote_median:,.0f}")
print(f"Ratio (should be ~1): {implied_dollar_adv / raw_quote_median:.3f}")
print(f"Panel has volume: {panel.has_field('volume')}")


In [ ]:
# Cell 5 — Stage 3: costs + liquidity (FTMO crypto CFD intraday stack).
# ShortBorrow deliberately absent: strategy is flat overnight by construction (no swap
# is the point of #011), crypto CFD shorts pay no intraday borrow, and ShortBorrow's
# per-bar charging (components.py:79-90) mis-annualizes ~70x on 30m bars.
from backtest.costs import Commission, Spread, MarketImpact, CompositeCostModel, LiquidityCap

costs = CompositeCostModel([
    Commission(bps=3.25),                              # per side, matches #009/#010 stack
    Spread(half_bps=5),                                # baseline; 00:00 UTC widening = later sensitivity
    MarketImpact(k=12, kind="sqrt", adv_lookback=20),  # trailing 10h of 30m-bar liquidity
])
liq = LiquidityCap(cap_pct=0.05, adv_lookback=20)      # 5% of median 30m bar volume

print("Cost components:", [type(m).__name__ for m in costs.models])
print(f"LiquidityCap: cap_pct={liq.cap_pct}, adv_lookback={liq.adv_lookback}")
print(f"Panel has volume: {panel.has_field('volume')}")


In [ ]:
# Cell 6 — Stage 4: single-path engine run (one-day warmup for the 20-bar ADV lookback).
from backtest.engine import Engine

WARMUP_BARS = 48  # one UTC day of 30m bars

engine = Engine(panel, strat, costs=costs, liquidity=liq, initial_capital=1_000_000)
result = engine.run(start=panel.dates[WARMUP_BARS])

print(f"Bars:          {len(result.equity_curve)}")
print(f"Rebalances:    {result.metadata['n_rebalances']}")
print(f"Date range:    {result.metadata['start']} -> {result.metadata['end']}")
print(f"Initial:       ${result.metadata['initial_capital']:>14,.0f}")
print(f"Final equity:  ${result.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return:  {(result.equity_curve.iloc[-1] / result.metadata['initial_capital'] - 1):+.2%}")
print(f"Total costs:   ${result.costs.sum():>14,.0f}")
print(f"Trading days w/ a position: {int((result.trades.abs().sum(axis=1) > 0).sum() / 2)} round trips approx")
print(f"Unfilled (liq-capped) total: ${result.unfilled.abs().sum().sum():,.0f}")
print(f"Max position weight:  {result.weights.abs().max().max():.2%}")
print(f"Max gross exposure:   {result.weights.abs().sum(axis=1).max():.2%}")


In [ ]:
# Cell 7 — Diagnostic (read-only): unintended overnight inventory + unfilled by year.
pos = result.positions["BTCUSDT"]
idx = pos.index
hm = idx.hour * 60 + idx.minute

# Intended exposure: only the 23:30->00:00 bar. Position recorded at the 23:30 stamp
# (post-trade) is carried into the midnight bar; position at any OTHER stamp should be 0.
intended_carry = (hm == 23 * 60 + 30)
held = pos.abs() > 1.0  # >$1 = held
overnight_bars = held & ~intended_carry & ~(hm == 0)  # held outside window (00:00 stamp itself is exit-bar, post-trade should be 0 too)
held_at_exit = held & (hm == 0)

pnl = result.pnl
pnl_intended = pnl[(hm == 0)].sum()              # last-half-hour P&L lands on the midnight bar
pnl_unintended = pnl[overnight_bars.shift(1, fill_value=False).values].sum()

print(f"Bars with intended position (23:30 stamps held): {int((held & intended_carry).sum())}")
print(f"Bars STILL holding at exit stamp (00:00, post-trade): {int(held_at_exit.sum())}")
print(f"Bars holding outside the window entirely: {int(overnight_bars.sum())}")
print(f"P&L on intended last-half-hour bars: ${pnl_intended:,.0f}")
print(f"P&L on bars following unintended holds: ${pnl_unintended:,.0f}")
print(f"Total costs: ${result.costs.sum():,.0f}")

print("\nBy year: unfilled $, year-end equity, bars-held-outside-window")
unf = result.unfilled.abs().sum(axis=1)
eq = result.equity_curve
for y in range(2017, 2027):
    m = idx.year == y
    if not m.any():
        continue
    print(f"  {y}: unfilled ${unf[m].sum():>13,.0f} | eq end ${eq[m].iloc[-1]:>11,.0f} "
          f"| overnight-bars {int(overnight_bars[m].sum()):>6}")


In [ ]:
# Cell 8 — Decay check: per-year gross signal quality on the held half-hours.
# Gross trade return = sign(signal) * last-half-hour BTC return, reconstructed from
# prices + the realized position sign at the 23:30 stamp (post-trade, pre-cost).
px = prices["BTCUSDT"]
stamps = px.index
hm = stamps.hour * 60 + stamps.minute

entry_stamps = stamps[hm == 23 * 60 + 30]
pos_sign = np.sign(result.positions["BTCUSDT"].reindex(entry_stamps).fillna(0.0))
exit_stamps = entry_stamps + pd.Timedelta("30min")
ret_lasthh = (px.reindex(exit_stamps).values / px.reindex(entry_stamps).values) - 1.0
ret_lasthh = pd.Series(ret_lasthh, index=entry_stamps)

trade_ret = pos_sign * ret_lasthh          # gross, per held half-hour, sign-only (unit notional)
traded = pos_sign != 0
tr = trade_ret[traded].dropna()

print(f"{'year':>5} {'trades':>7} {'hit%':>6} {'mean bps':>9} {'gross Sh(ann365)':>17}")
for y in range(2017, 2027):
    s = tr[tr.index.year == y]
    if len(s) == 0:
        continue
    sh = s.mean() / s.std() * np.sqrt(365) if s.std() > 0 else float("nan")
    print(f"{y:>5} {len(s):>7} {100 * (s > 0).mean():>6.1f} {1e4 * s.mean():>9.2f} {sh:>17.2f}")

sh_all = tr.mean() / tr.std() * np.sqrt(365)
print(f"\nALL  : {len(tr)} trades | hit {100 * (tr > 0).mean():.1f}% | "
      f"mean {1e4 * tr.mean():+.2f} bps/trade | gross Sharpe {sh_all:+.2f}")
print(f"Cost gate for reference: ~16.5 bps/round trip at FTMO stack")


In [ ]:
# Cell 9 — Trial 2 (pre-registered queue sketch): rest-of-day variant.
# Same signal (sign of 00:00->00:30 return); hold 00:30->24:00 instead of the last half-hour.
# Enter at the 00:30 bar, exit at the 00:00 (midnight) bar. K=2 for DSR at Stage 9.
def _rod_rebalance_bars(dates):
    hm = dates.hour * 60 + dates.minute
    return dates[(hm == 30) | (hm == 0)]


class IntradayTSMRestOfDay(Strategy):
    """Queue #011 sketch: first-half-hour sign held for the rest of the UTC day."""
    rebalance_frequency = staticmethod(_rod_rebalance_bars)

    def __init__(self, gross: float = 0.5):
        self.gross = gross
        self.risk = RiskConfig(max_position=gross, max_gross=1.0)

    def __repr__(self):
        return f"IntradayTSMRestOfDay(signal=00:00-00:30, trade=00:30-24:00, gross={self.gross})"

    def generate_weights(self, data, t):
        zero = pd.Series(0.0, index=data.assets)
        if not (t.hour == 0 and t.minute == 30):
            return zero                                  # midnight bar: target flat
        day = t.normalize()
        px = data.prices["BTCUSDT"]
        try:
            p_mid = px.at[day]
        except KeyError:
            return zero
        p_0030 = px.at[t]                                # t IS the 00:30 stamp, in-view
        if pd.isna(p_mid) or pd.isna(p_0030) or p_0030 == p_mid:
            return zero
        return pd.Series({"BTCUSDT": self.gross * np.sign(p_0030 / p_mid - 1.0)})


strat_rod = IntradayTSMRestOfDay(gross=0.5)
print(strat_rod)
rb = strat_rod.rebalance_dates(panel.dates)
print(f"Rebalance bars: {len(rb)} (expect ~2/day = {2 * 3220})")


In [ ]:
# Cell 10 — Stage 4 run, trial 2 (rest-of-day variant). Same costs/liq/capital as Cell 6.
engine_rod = Engine(panel, strat_rod, costs=costs, liquidity=liq, initial_capital=1_000_000)
result_rod = engine_rod.run(start=panel.dates[WARMUP_BARS])

print(f"Bars:          {len(result_rod.equity_curve)}")
print(f"Rebalances:    {result_rod.metadata['n_rebalances']}")
print(f"Final equity:  ${result_rod.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return:  {(result_rod.equity_curve.iloc[-1] / 1_000_000 - 1):+.2%}")
print(f"Total costs:   ${result_rod.costs.sum():>14,.0f}")
print(f"Unfilled (liq-capped) total: ${result_rod.unfilled.abs().sum().sum():,.0f}")
print(f"Max position weight:  {result_rod.weights.abs().max().max():.2%}")
print(f"Max gross exposure:   {result_rod.weights.abs().sum(axis=1).max():.2%}")

# Gross per-trade quality, same construction as Cell 8, for the held window 00:30->24:00
stamps_ = prices["BTCUSDT"].index
hm_ = stamps_.hour * 60 + stamps_.minute
entry_ = stamps_[hm_ == 30]
sign_ = np.sign(result_rod.positions["BTCUSDT"].reindex(entry_).fillna(0.0))
exit_ = entry_.normalize() + pd.Timedelta("1D")
px_ = prices["BTCUSDT"]
ret_ = pd.Series(px_.reindex(exit_).values / px_.reindex(entry_).values - 1.0, index=entry_)
tr_ = (sign_ * ret_)[sign_ != 0].dropna()
sh_ = tr_.mean() / tr_.std() * np.sqrt(365)
print(f"\nGross: {len(tr_)} trades | hit {100 * (tr_ > 0).mean():.1f}% | "
      f"mean {1e4 * tr_.mean():+.2f} bps/trade | gross Sharpe {sh_:+.2f} (vs cost gate ~16.5bp)")
